# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")


✅ Libraries imported successfully!
Boto3 version: 1.37.3
Rasterio version: 1.4.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [4]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [5]:

EVENT_NAME = '202405_Heat_TX'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'ECOSTRESS'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [6]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [7]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

✅ S3 client initialized successfully
   Found 68 accessible buckets
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 91 .tif files in the S3 bucket.


['drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023126133547_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023127043620_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023127043712_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023131025921_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023131030013_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023134021109_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023135012241_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001

## Configure bucket and paths (no need to create session manually)

In [8]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

## Define Chunked COG Conversion Function

This function handles the conversion of files to Cloud Optimized GeoTIFFs with:
- Chunked processing to handle large files
- Memory monitoring
- Progress tracking
- Proper CRS and caching

In [9]:
# Check current cache status using the imported function
check_cache_status()

📁 Cache directory does not exist: data_download/
   Creating cache directory...
✅ Cache directory created: data_download/


(0, 0)

In [10]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    _
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

# Process files

In [11]:
keys

['drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023126133547_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023127043620_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023127043712_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023131025921_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023131030013_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023134021109_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023135012241_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001

In [22]:
def create_cog_filename_ecostress(f, EVENT_NAME):
    """Convert ECOSTRESS filename to new format with datetime from doy pattern."""
    from datetime import datetime, timedelta
    import re
    from pathlib import Path
    
    path = Path(f)
    filename = path.stem
    extension = path.suffix
    
    # Check if it's a control data file
    is_control = 'ControlData' in str(path)
    
    # Extract the doy pattern (e.g., doy2023126133547)
    doy_pattern = r'doy(\d{4})(\d{3})(\d{6})'
    match = re.search(doy_pattern, filename)
    
    if match:
        year = int(match.group(1))
        doy = int(match.group(2))
        time_str = match.group(3)
        
        # Convert DOY to date
        date = datetime(year, 1, 1) + timedelta(days=doy - 1)
        
        # Parse time HHMMSS
        hour = time_str[0:2]
        minute = time_str[2:4]
        second = time_str[4:6]
        
        # Format datetime as YYYYMMDDTHH:MM:SSZ
        formatted_datetime = f"{date.strftime('%Y-%m-%d')}T{hour}:{minute}:{second}Z"
        
        # Extract the product parts (e.g., ECO2LSTE.001_SDS_LST or ECO2LSTE.001_SDS_LST_err)
        product_match = re.search(r'(ECO2LSTE\.001_SDS_[A-Z]+(?:_err)?)', filename)
        if product_match:
            product_part = product_match.group(1)
        else:
            product_part = 'ECO2LSTE.001'
        
        # Extract aid number
        aid_match = re.search(r'aid(\d+)', filename)
        if aid_match:
            aid_part = f"aid{aid_match.group(1)}"
        else:
            aid_part = "aid0001"
        
        # Build new filename
        if is_control:
            cog_filename = f'{EVENT_NAME}_ControlData_{product_part}_{aid_part}_{formatted_datetime}{extension}'
        else:
            cog_filename = f'{EVENT_NAME}_{product_part}_{aid_part}_{formatted_datetime}{extension}'
    else:
        # Handle files without doy pattern (like additional.tif)
        if is_control:
            cog_filename = f'{EVENT_NAME}_ControlData_{filename}{extension}'
        else:
            cog_filename = f'{EVENT_NAME}_{filename}{extension}'
    
    return cog_filename

filter_str = 'ControlData'

pattern = re.compile(r'ControlData.*LST_doy.*\.tif$')
filter_ =  [f for f in keys if pattern.search(f)]

# Test functions
print("Testing WM filename:")
# filter_ = [i for i in keys if filter_]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_ecostress(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-06T13:35:47Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-07T04:36:20Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-07T04:37:12Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-11T02:59:21Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-11T03:00:13Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-14T02:11:09Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-15T01:22:41Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-15T01:23:33Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-15T09:32:30Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-15T09:33:22Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-18T08:43:10Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-

In [23]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename_ecostress, 
                                target_dir = "ECOSTRESS/LST", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-06T13:35:47Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-07T04:36:20Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-07T04:37:12Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-11T02:59:21Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-11T03:00:13Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-14T02:11:09Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-15T01:22:41Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-15T01:23:33Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-15T09:32:30Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-15T09:33:22Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-18T08:43:10Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-18

Reading input: /tmp/tmpg89up5q0_temp.tif



   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpze9j1hrq.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/LST/202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-06T13:35:47Z.tif
   [MEMORY] Final: 346.0 MB (Change: +49.1 MB)


...

   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/LST/202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-29T19:41:31Z.tif
   [MEMORY] Final: 404.9 MB (Change: -51.2 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-29T19:41:31Z.tif

[19/19] Processing: drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023150035150_aid0001.tif
   Output filename: 202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-30T03:51:50Z.tif
   [MEMORY] Initial: 404.9 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected d

Reading input: /tmp/tmpv1xqp0cj_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpo7ubjr2b.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/LST/202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-30T03:51:50Z.tif
   [MEMORY] Final: 409.6 MB (Change: +4.8 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_aid0001_2023-05-30T03:51:50Z.tif

✅ Batch processing complete: 19 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/LST/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/LST/files_converted.csv
📁 COGs saved locally to: output/202405_Heat_TX

📊 BATCH PROCESSING SUMMARY
Total files processed: 19
Successful: 19
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-09T21:29:50.034382


In [24]:
keys

['drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023126133547_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023127043620_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023127043712_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023131025921_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023131030013_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023134021109_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023135012241_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001

In [29]:
def create_cog_filename_ecostress(f, EVENT_NAME):
    """Convert ECOSTRESS filename to new format with datetime from doy pattern."""
    from datetime import datetime, timedelta
    import re
    from pathlib import Path
    
    path = Path(f)
    filename = path.stem
    extension = path.suffix
    
    # Extract the doy pattern (e.g., doy2023126133547)
    doy_pattern = r'doy(\d{4})(\d{3})(\d{6})'
    match = re.search(doy_pattern, filename)
    
    if match:
        year = int(match.group(1))
        doy = int(match.group(2))
        time_str = match.group(3)
        
        # Convert DOY to date
        date = datetime(year, 1, 1) + timedelta(days=doy - 1)
        
        # Parse time HHMMSS
        hour = time_str[0:2]
        minute = time_str[2:4]
        second = time_str[4:6]
        
        # Format datetime as YYYYMMDDTHH:MM:SSZ
        formatted_datetime = f"{date.strftime('%Y-%m-%d')}T{hour}:{minute}:{second}Z"
        
        # Extract the product parts (e.g., ECO2LSTE.001_SDS_LST or ECO2LSTE.001_SDS_LST_err)
        product_match = re.search(r'(ECO2LSTE\.001_SDS_[A-Z]+(?:_err)?)', filename)
        if product_match:
            product_part = product_match.group(1)
        else:
            product_part = 'ECO2LSTE.001'
        
        # Extract aid number
        aid_match = re.search(r'aid(\d+)', filename)
        if aid_match:
            aid_part = f"aid{aid_match.group(1)}"
        else:
            aid_part = "aid0001"
        
        # Build new filename - no ControlData label for non-control files
        cog_filename = f'{EVENT_NAME}_{product_part}_{aid_part}_{formatted_datetime}{extension}'
    else:
        # Handle files without doy pattern (like additional.tif)
        cog_filename = f'{EVENT_NAME}_{filename}{extension}'
    
    return cog_filename


pattern = re.compile(r'^(?!.*ControlData).*LST_doy.*\.tif$')
filter_ =  [f for f in keys if pattern.search(f)]

# Test functions
print("Testing WM filename:")
# filter_ = [i for i in keys if filter_]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_ecostress(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-15T08:37:21Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-15T08:38:13Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-17T23:35:53Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-17T23:36:45Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-18T22:46:44Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-22T06:02:03Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-22T06:02:55Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-24T20:59:30Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-25T20:10:46Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-25T20:11:38Z.tif


In [30]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename_ecostress, 
                                target_dir = "ECOSTRESS/LST", 
                                EVENT_NAME = EVENT_NAME)

Testing filenames:
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-15T08:37:21Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-15T08:38:13Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-17T23:35:53Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-17T23:36:45Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-18T22:46:44Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-22T06:02:03Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-22T06:02:55Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-24T20:59:30Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-25T20:10:46Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-25T20:11:38Z.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202405_Heat_TX/ECOSTRESS
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/ECOSTRESS/LST

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/20240

Reading input: /tmp/tmp0ux5jpz0_temp.tif

Updating dataset tags...
Writing output to: /tmp/tmpmxod8qmf.tif


   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=0, center sample non-zero=0/82944
            Estimated data coverage: 37.4% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint16
   [NODATA] Using nodata value 0 for uint16 data
   [PREDICTOR] Data type: uint16, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - No overviews found
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/LST/202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-15T08:37:21Z.tif
   [MEMORY] Final: 417.9 MB (Change: +1.8 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved C

Reading input: /tmp/tmpwi3cc53h_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


...

   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/LST/202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-25T20:10:46Z.tif
   [MEMORY] Final: 413.6 MB (Change: -40.1 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-25T20:10:46Z.tif

[10/10] Processing: drcs_activations/202405_Heat_TX/ECOSTRESS/ECO2LSTE.001_SDS_LST_doy2024146201138_aid0001.tif
   Output filename: 202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-25T20:11:38Z.tif
   [MEMORY] Initial: 413.6 MB
   [DOWNLOAD] Downloading from S3...


Reading input: /tmp/tmp9e895_db_temp.tif



   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=15706, center sample non-zero=13998/1000000
            Estimated data coverage: 20.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint16
   [NODATA] Using nodata value 0 for uint16 data
   [PREDICTOR] Data type: uint16, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpshh_p75e.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/LST/202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-25T20:11:38Z.tif
   [MEMORY] Final: 477.5 MB (Change: +63.9 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202405_Heat_TX_ECO2LSTE.001_SDS_LST_aid0001_2024-05-25T20:11:38Z.tif

✅ Batch processing complete: 10 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/LST/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/LST/files_converted.csv
📁 COGs saved locally to: output/202405_Heat_TX

📊 BATCH PROCESSING SUMMARY
Total files processed: 10
Successful: 10
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-09T21:34:43.162484


In [31]:
def create_cog_filename_ecostress(f, EVENT_NAME):
    """Convert ECOSTRESS filename to new format with datetime from doy pattern."""
    from datetime import datetime, timedelta
    import re
    from pathlib import Path
    
    path = Path(f)
    filename = path.stem
    extension = path.suffix
    
    # Check if it's a control data file
    is_control = 'ControlData' in str(path)
    
    # Extract the doy pattern (e.g., doy2023126133547)
    doy_pattern = r'doy(\d{4})(\d{3})(\d{6})'
    match = re.search(doy_pattern, filename)
    
    if match:
        year = int(match.group(1))
        doy = int(match.group(2))
        time_str = match.group(3)
        
        # Convert DOY to date
        date = datetime(year, 1, 1) + timedelta(days=doy - 1)
        
        # Parse time HHMMSS
        hour = time_str[0:2]
        minute = time_str[2:4]
        second = time_str[4:6]
        
        # Format datetime as YYYYMMDDTHH:MM:SSZ
        formatted_datetime = f"{date.strftime('%Y-%m-%d')}T{hour}:{minute}:{second}Z"
        
        # Extract the product parts (e.g., ECO2LSTE.001_SDS_LST or ECO2LSTE.001_SDS_LST_err)
        product_match = re.search(r'(ECO2LSTE\.001_SDS_[A-Z]+(?:_err)?)', filename)
        if product_match:
            product_part = product_match.group(1)
        else:
            product_part = 'ECO2LSTE.001'
        
        # Extract aid number
        aid_match = re.search(r'aid(\d+)', filename)
        if aid_match:
            aid_part = f"aid{aid_match.group(1)}"
        else:
            aid_part = "aid0001"
        
        # Build new filename
        if is_control:
            cog_filename = f'{EVENT_NAME}_ControlData_{product_part}_{aid_part}_{formatted_datetime}{extension}'
        else:
            cog_filename = f'{EVENT_NAME}_{product_part}_{aid_part}_{formatted_datetime}{extension}'
    else:
        # Handle files without doy pattern (like additional.tif)
        if is_control:
            cog_filename = f'{EVENT_NAME}_ControlData_{filename}{extension}'
        else:
            cog_filename = f'{EVENT_NAME}_{filename}{extension}'
    
    return cog_filename

filter_str = 'ControlData'

pattern = re.compile(r'ControlData.*LST_err_doy.*\.tif$')
filter_ =  [f for f in keys if pattern.search(f)]

# Test functions
print("Testing WM filename:")
# filter_ = [i for i in keys if filter_]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_ecostress(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-06T13:35:47Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-07T04:36:20Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-07T04:37:12Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-10T11:58:45Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-11T02:59:21Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-11T03:00:13Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-14T02:11:09Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-14T10:21:04Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-15T01:22:41Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-15T01:23:33Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-15T09:32:30Z.tif
  202405_Heat_TX_Contr

In [32]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename_ecostress, 
                                target_dir = "ECOSTRESS/LST_err", 
                                EVENT_NAME = EVENT_NAME)

Reading input: /tmp/tmp6knxyaw2_temp.tif



Testing filenames:
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-06T13:35:47Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-07T04:36:20Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-07T04:37:12Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-10T11:58:45Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-11T02:59:21Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-11T03:00:13Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-14T02:11:09Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-14T10:21:04Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-15T01:22:41Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-15T01:23:33Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-15T09:32:30Z.tif
  202405_Heat_TX_Control

Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpglwjqi9l.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/LST_err/202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-06T13:35:47Z.tif
   [MEMORY] Final: 410.7 MB (Change: -66.7 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-06T13:35:47Z.tif

[2/21] Processing: drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_err_doy2023127043620_aid0001.tif
   Output filename: 202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-07T04:36:20Z.tif
   [MEMORY] Initial: 410.7 MB
   [DOWNLOAD] Downloading from S3...


Reading input: /tmp/tmpenvfi1vp_temp.tif



...

   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/LST_err/202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-29T19:41:31Z.tif
   [MEMORY] Final: 450.9 MB (Change: +0.3 MB)


Reading input: /tmp/tmplgwwypgf_temp.tif


✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-29T19:41:31Z.tif

[21/21] Processing: drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_err_doy2023150035150_aid0001.tif
   Output filename: 202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-30T03:51:50Z.tif
   [MEMORY] Initial: 450.9 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=62, max=75, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary Geo


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp64di_3b_.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/LST_err/202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-30T03:51:50Z.tif
   [MEMORY] Final: 450.9 MB (Change: +0.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_LST_err_aid0001_2023-05-30T03:51:50Z.tif

✅ Batch processing complete: 21 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/LST_err/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/LST_err/files_converted.csv
📁 COGs saved locally to: output/202405_Heat_TX

📊 BATCH PROCESSING SUMMARY
Total files processed: 21
Successful: 21
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-09T21:35:53.

In [33]:
def create_cog_filename_ecostress(f, EVENT_NAME):
    """Convert ECOSTRESS filename to new format with datetime from doy pattern."""
    from datetime import datetime, timedelta
    import re
    from pathlib import Path
    
    path = Path(f)
    filename = path.stem
    extension = path.suffix
    
    # Extract the doy pattern (e.g., doy2023126133547)
    doy_pattern = r'doy(\d{4})(\d{3})(\d{6})'
    match = re.search(doy_pattern, filename)
    
    if match:
        year = int(match.group(1))
        doy = int(match.group(2))
        time_str = match.group(3)
        
        # Convert DOY to date
        date = datetime(year, 1, 1) + timedelta(days=doy - 1)
        
        # Parse time HHMMSS
        hour = time_str[0:2]
        minute = time_str[2:4]
        second = time_str[4:6]
        
        # Format datetime as YYYYMMDDTHH:MM:SSZ
        formatted_datetime = f"{date.strftime('%Y-%m-%d')}T{hour}:{minute}:{second}Z"
        
        # Extract the product parts (e.g., ECO2LSTE.001_SDS_LST or ECO2LSTE.001_SDS_LST_err)
        product_match = re.search(r'(ECO2LSTE\.001_SDS_[A-Z]+(?:_err)?)', filename)
        if product_match:
            product_part = product_match.group(1)
        else:
            product_part = 'ECO2LSTE.001'
        
        # Extract aid number
        aid_match = re.search(r'aid(\d+)', filename)
        if aid_match:
            aid_part = f"aid{aid_match.group(1)}"
        else:
            aid_part = "aid0001"
        
        # Build new filename - no ControlData label for non-control files
        cog_filename = f'{EVENT_NAME}_{product_part}_{aid_part}_{formatted_datetime}{extension}'
    else:
        # Handle files without doy pattern (like additional.tif)
        cog_filename = f'{EVENT_NAME}_{filename}{extension}'
    
    return cog_filename


pattern = re.compile(r'^(?!.*ControlData).*LST_err_doy.*\.tif$')
filter_ =  [f for f in keys if pattern.search(f)]

# Test functions
print("Testing WM filename:")
# filter_ = [i for i in keys if filter_]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_ecostress(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-15T08:37:21Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-15T08:38:13Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-17T23:35:53Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-17T23:36:45Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-18T22:46:44Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-22T06:02:03Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-22T06:02:55Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-24T20:59:30Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-25T20:10:46Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-25T20:11:38Z.tif


In [34]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename_ecostress, 
                                target_dir = "ECOSTRESS/LST_err", 
                                EVENT_NAME = EVENT_NAME)

Reading input: /tmp/tmpz2xkrc1y_temp.tif

Updating dataset tags...
Writing output to: /tmp/tmpbxcmszby.tif


Testing filenames:
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-15T08:37:21Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-15T08:38:13Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-17T23:35:53Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-17T23:36:45Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-18T22:46:44Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-22T06:02:03Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-22T06:02:55Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-24T20:59:30Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-25T20:10:46Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-25T20:11:38Z.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202405_Heat_TX/ECOSTRESS
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/ECOSTRESS/LST_err

🌊 Processing Files (Chunked)


Reading input: /tmp/tmprs5t5tx5_temp.tif



   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=48, center sample non-zero=500143/1000000
            Estimated data coverage: 49.9% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpbnwok97y.tif


...

   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/LST_err/202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-25T20:10:46Z.tif
   [MEMORY] Final: 451.2 MB (Change: +0.0 MB)


Reading input: /tmp/tmp4rrl_jdj_temp.tif



✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-25T20:10:46Z.tif

[10/10] Processing: drcs_activations/202405_Heat_TX/ECOSTRESS/ECO2LSTE.001_SDS_LST_err_doy2024146201138_aid0001.tif
   Output filename: 202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-25T20:11:38Z.tif
   [MEMORY] Initial: 451.2 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=60, center sample non-zero=13998/1000000
            Estimated data coverage: 20.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIF

Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp6wz5t2yl.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/LST_err/202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-25T20:11:38Z.tif
   [MEMORY] Final: 451.2 MB (Change: +0.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202405_Heat_TX_ECO2LSTE.001_SDS_LST_err_aid0001_2024-05-25T20:11:38Z.tif

✅ Batch processing complete: 10 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/LST_err/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/LST_err/files_converted.csv
📁 COGs saved locally to: output/202405_Heat_TX

📊 BATCH PROCESSING SUMMARY
Total files processed: 10
Successful: 10
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-09T21:36:35.451049


In [35]:
def create_cog_filename_ecostress(f, EVENT_NAME):
    """Convert ECOSTRESS filename to new format with datetime from doy pattern."""
    from datetime import datetime, timedelta
    import re
    from pathlib import Path
    
    path = Path(f)
    filename = path.stem
    extension = path.suffix
    
    # Check if it's a control data file
    is_control = 'ControlData' in str(path)
    
    # Extract the doy pattern (e.g., doy2023126133547)
    doy_pattern = r'doy(\d{4})(\d{3})(\d{6})'
    match = re.search(doy_pattern, filename)
    
    if match:
        year = int(match.group(1))
        doy = int(match.group(2))
        time_str = match.group(3)
        
        # Convert DOY to date
        date = datetime(year, 1, 1) + timedelta(days=doy - 1)
        
        # Parse time HHMMSS
        hour = time_str[0:2]
        minute = time_str[2:4]
        second = time_str[4:6]
        
        # Format datetime as YYYYMMDDTHH:MM:SSZ
        formatted_datetime = f"{date.strftime('%Y-%m-%d')}T{hour}:{minute}:{second}Z"
        
        # Extract the product parts (e.g., ECO2LSTE.001_SDS_LST or ECO2LSTE.001_SDS_LST_err)
        product_match = re.search(r'(ECO2LSTE\.001_SDS_[A-Z]+(?:_err)?)', filename)
        if product_match:
            product_part = product_match.group(1)
        else:
            product_part = 'ECO2LSTE.001'
        
        # Extract aid number
        aid_match = re.search(r'aid(\d+)', filename)
        if aid_match:
            aid_part = f"aid{aid_match.group(1)}"
        else:
            aid_part = "aid0001"
        
        # Build new filename
        if is_control:
            cog_filename = f'{EVENT_NAME}_ControlData_{product_part}_{aid_part}_{formatted_datetime}{extension}'
        else:
            cog_filename = f'{EVENT_NAME}_{product_part}_{aid_part}_{formatted_datetime}{extension}'
    else:
        # Handle files without doy pattern (like additional.tif)
        if is_control:
            cog_filename = f'{EVENT_NAME}_ControlData_{filename}{extension}'
        else:
            cog_filename = f'{EVENT_NAME}_{filename}{extension}'
    
    return cog_filename


pattern = re.compile(r'ControlData.*QC_doy.*\.tif$')
filter_ =  [f for f in keys if pattern.search(f)]

# Test functions
print("Testing WM filename:")
# filter_ = [i for i in keys if filter_]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_ecostress(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-06T13:35:47Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-07T04:36:20Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-07T04:37:12Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-10T11:58:45Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-11T02:59:21Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-11T03:00:13Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-14T02:11:09Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-14T10:21:04Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-15T01:22:41Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-15T01:23:33Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-15T09:32:30Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-15T09:33:22Z

In [36]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename_ecostress, 
                                target_dir = "ECOSTRESS/QC", 
                                EVENT_NAME = EVENT_NAME)

Testing filenames:
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-06T13:35:47Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-07T04:36:20Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-07T04:37:12Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-10T11:58:45Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-11T02:59:21Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-11T03:00:13Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-14T02:11:09Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-14T10:21:04Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-15T01:22:41Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-15T01:23:33Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-15T09:32:30Z.tif
  202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-15T09:33:22Z.t

Reading input: /tmp/tmppmo8vz1c_temp.tif



   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=-99999, max=-99999, center sample non-zero=0/1000000
            Estimated data coverage: 19.1% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: int32
   [NODATA] Using nodata value -9999 for int32 data
   [PREDICTOR] Data type: int32, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp5kc86di5.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/QC/202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-06T13:35:47Z.tif
   [MEMORY] Final: 430.1 MB (Change: -21.1 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-06T13:35:47Z.tif

[2/21] Processing: drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_QC_doy2023127043620_aid0001.tif
   Output filename: 202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-07T04:36:20Z.tif
   [MEMORY] Initial: 430.1 MB
   [DOWNLOAD] Downloading from S3...


...

   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/QC/202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-29T19:41:31Z.tif
   [MEMORY] Final: 522.8 MB (Change: -39.1 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-29T19:41:31Z.tif

[21/21] Processing: drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_QC_doy2023150035150_aid0001.tif
   Output filename: 202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-30T03:51:50Z.tif
   [MEMORY] Initial: 522.8 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data..

Reading input: /tmp/tmpti768a67_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpc6st571y.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/QC/202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-30T03:51:50Z.tif
   [MEMORY] Final: 522.8 MB (Change: +0.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202405_Heat_TX_ControlData_ECO2LSTE.001_SDS_QC_aid0001_2023-05-30T03:51:50Z.tif

✅ Batch processing complete: 21 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/QC/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/QC/files_converted.csv
📁 COGs saved locally to: output/202405_Heat_TX

📊 BATCH PROCESSING SUMMARY
Total files processed: 21
Successful: 21
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-09T21:38:22.477429


In [40]:
keys

['drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023126133547_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023127043620_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023127043712_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023131025921_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023131030013_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023134021109_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023135012241_aid0001.tif',
 'drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001

In [41]:
def create_cog_filename_ecostress(f, EVENT_NAME):
    """Convert ECOSTRESS filename to new format with datetime from doy pattern."""
    from datetime import datetime, timedelta
    import re
    from pathlib import Path
    
    path = Path(f)
    filename = path.stem
    extension = path.suffix
    
    # Extract the doy pattern (e.g., doy2023126133547)
    doy_pattern = r'doy(\d{4})(\d{3})(\d{6})'
    match = re.search(doy_pattern, filename)
    
    if match:
        year = int(match.group(1))
        doy = int(match.group(2))
        time_str = match.group(3)
        
        # Convert DOY to date
        date = datetime(year, 1, 1) + timedelta(days=doy - 1)
        
        # Parse time HHMMSS
        hour = time_str[0:2]
        minute = time_str[2:4]
        second = time_str[4:6]
        
        # Format datetime as YYYYMMDDTHH:MM:SSZ
        formatted_datetime = f"{date.strftime('%Y-%m-%d')}T{hour}:{minute}:{second}Z"
        
        # Extract the product parts (e.g., ECO2LSTE.001_SDS_LST or ECO2LSTE.001_SDS_LST_err)
        product_match = re.search(r'(ECO2LSTE\.001_SDS_[A-Z]+(?:_err)?)', filename)
        if product_match:
            product_part = product_match.group(1)
        else:
            product_part = 'ECO2LSTE.001'
        
        # Extract aid number
        aid_match = re.search(r'aid(\d+)', filename)
        if aid_match:
            aid_part = f"aid{aid_match.group(1)}"
        else:
            aid_part = "aid0001"
        
        # Build new filename - no ControlData label for non-control files
        cog_filename = f'{EVENT_NAME}_{product_part}_{aid_part}_{formatted_datetime}{extension}'
    else:
        # Handle files without doy pattern (like additional.tif)
        cog_filename = f'{EVENT_NAME}_{filename}{extension}'
    
    return cog_filename


pattern = re.compile(r'^(?!.*ControlData).*QC_doy.*\.tif$')
filter_ =  [f for f in keys if pattern.search(f)]

# Test functions
print("Testing WM filename:")
# filter_ = [i for i in keys if filter_]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_ecostress(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-15T08:37:21Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-15T08:38:13Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-17T23:35:53Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-17T23:36:45Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-22T06:02:03Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-22T06:02:55Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-24T20:59:30Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-25T20:10:46Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-25T20:11:38Z.tif


In [42]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename_ecostress, 
                                target_dir = "ECOSTRESS/QC", 
                                EVENT_NAME = EVENT_NAME)

Reading input: /tmp/tmpghzmpa_n_temp.tif



Testing filenames:
  202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-15T08:37:21Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-15T08:38:13Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-17T23:35:53Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-17T23:36:45Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-22T06:02:03Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-22T06:02:55Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-24T20:59:30Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-25T20:10:46Z.tif
  202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-25T20:11:38Z.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202405_Heat_TX/ECOSTRESS
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/ECOSTRESS/QC

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202405_Heat_TX

[1/9] Processing: drcs_activations/202405_Heat_TX/ECOSTRESS/ECO2LSTE.0

Updating dataset tags...
Writing output to: /tmp/tmp7bchhwmi.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - No overviews found
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/QC/202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-15T08:37:21Z.tif
   [MEMORY] Final: 522.8 MB (Change: +0.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-15T08:37:21Z.tif

[2/9] Processing: drcs_activations/202405_Heat_TX/ECOSTRESS/ECO2LSTE.001_SDS_QC_doy2024136083813_aid0001.tif
   Output filename: 202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-15T08:38:13Z.tif
   [MEMORY] Initial: 522.8 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=-99999, max=36549, c

Reading input: /tmp/tmp_4lgllpa_temp.tif



...

   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/QC/202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-25T20:10:46Z.tif
   [MEMORY] Final: 473.5 MB (Change: +26.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-25T20:10:46Z.tif

[9/9] Processing: drcs_activations/202405_Heat_TX/ECOSTRESS/ECO2LSTE.001_SDS_QC_doy2024146201138_aid0001.tif
   Output filename: 202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-25T20:11:38Z.tif
   [MEMORY] Initial: 473.5 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=-99999, max=19904, center sample non-zero=1399

Reading input: /tmp/tmp62kysklv_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpumzu10tl.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/QC/202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-25T20:11:38Z.tif
   [MEMORY] Final: 535.8 MB (Change: +62.3 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202405_Heat_TX_ECO2LSTE.001_SDS_QC_aid0001_2024-05-25T20:11:38Z.tif

✅ Batch processing complete: 9 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/QC/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/ECOSTRESS/QC/files_converted.csv
📁 COGs saved locally to: output/202405_Heat_TX

📊 BATCH PROCESSING SUMMARY
Total files processed: 9
Successful: 9
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-09T21:39:21.320758


## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.

## Memory Usage Summary

You can check the final memory usage and cleanup

In [43]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")


📊 Memory Usage Summary:
  Current memory usage: 535.8 MB
  Available memory: 26541.7 MB
  Memory percent used: 16.1%
